# UTTA-Med — Gate C salvage (NO retrain)

Settings: **GPU T4**, Internet on.

**Add Input** the dataset that contains `camelyon17_resnet18_source_s42_BEST.pt` (epoch 3, val AUROC 0.9793).

This notebook does **not** train. It searches a fair Tent/EATA on **OOD val (hospital 1)** then runs test once.

In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow
import torch
assert torch.cuda.is_available(), 'Enable GPU T4'
print(torch.cuda.get_device_name(0))

In [ ]:
import json, random, copy
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')

def find_ckpt():
    cands = list(Path('/kaggle/input').rglob('*.pt')) + list(WORK.glob('*.pt'))
    for p in cands:
        try:
            st = torch.load(p, map_location='cpu', weights_only=False)
            print('found', p, 'epoch', st.get('epoch'), 'val_auroc', st.get('val_auroc'))
            if int(st.get('epoch', -1)) == 3 and float(st.get('val_auroc', 0)) >= 0.978:
                return p, st
        except Exception as e:
            print('skip', p, e)
    raise FileNotFoundError('Add Input the BEST epoch-3 checkpoint (val_auroc ~0.979)')

CKPT_PATH, CKPT = find_ckpt()
print('USING', CKPT_PATH)

In [ ]:
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf = transforms.Compose([transforms.Resize((96,96)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
def cid(v): return int(v.item() if hasattr(v,'item') else v)
class HFCamelyon(Dataset):
    def __init__(self, split, centers):
        centers={int(c) for c in centers}
        keep=[i for i,c in enumerate(split['center']) if cid(c) in centers]
        self.ds = split.select(keep) if len(keep)<len(split) else split
        print(' kept', len(self.ds), sorted(centers))
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        r=self.ds[i]; img=r['image']
        img = img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(np.array(img)).convert('RGB')
        return tf(img), torch.tensor(float(int(r['label'])))
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        bb=models.resnet18(weights=None); d=bb.fc.in_features; bb.fc=nn.Identity(); self.backbone=bb
        self.classifier=nn.Sequential(nn.Linear(d,d), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(d,1))
    def forward(self,x): return self.classifier(self.backbone(x))

raw=load_dataset('wltjr1007/Camelyon17-WILDS')
ood_set=HFCamelyon(raw['validation'],{1})
test_set=HFCamelyon(raw['test'],{2})
ood_loader=DataLoader(ood_set,batch_size=64,shuffle=False,num_workers=0,pin_memory=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=False,num_workers=0,pin_memory=True)

In [ ]:
def ece_score(y,p,n=15):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); bins=np.linspace(0,1,n+1); e=0.0
    for i in range(n):
        m=(p>bins[i])&(p<=bins[i+1]) if i else (p>=bins[i])&(p<=bins[i+1])
        if m.any(): e += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(e)
def metrics_of(y,p):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); pred=(p>=0.5).astype(int)
    return dict(n=int(len(y)), accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)), recall=float(recall_score(y,pred,zero_division=0)),
        sensitivity=float(recall_score(y,pred,zero_division=0)),
        specificity=float(((y==0)&(pred==0)).sum()/max((y==0).sum(),1)),
        f1=float(f1_score(y,pred,zero_division=0)), auroc=float(roc_auc_score(y,p)),
        auprc=float(average_precision_score(y,p)), ece=ece_score(y,p), brier=float(np.mean((p-y)**2)))
def flip_table(y,b,a):
    y=np.asarray(y).ravel().astype(int); b=(np.asarray(b).ravel()>=0.5).astype(int); a=(np.asarray(a).ravel()>=0.5).astype(int)
    return dict(correction_rate=float(((b!=y)&(a==y)).sum()/max((b!=y).sum(),1)),
                harm_rate=float(((b==y)&(a!=y)).sum()/max((b==y).sum(),1)), flip_rate=float((a!=b).mean()),
                correction=int(((b!=y)&(a==y)).sum()), harmful=int(((b==y)&(a!=y)).sum()))
@torch.no_grad()
def evaluate(m,ld):
    m.eval(); ys,ps=[],[]
    for x,y in ld:
        x=x.to(device,non_blocking=True); ps.append(torch.sigmoid(m(x)).view(-1).cpu().numpy()); ys.append(y.view(-1).numpy())
    return metrics_of(np.concatenate(ys), np.concatenate(ps))

def load_fresh():
    m=Net().to(device); m.load_state_dict(CKPT['model']); return m

src=load_fresh(); src.eval()
ood_src=evaluate(src, ood_loader)
print('ASSERT OOD source AUROC', round(ood_src['auroc'],4), 'expect ~0.9793')
assert ood_src['auroc'] >= 0.975, 'Wrong checkpoint — stop'

In [ ]:
def bn_affine(m):
    for p in m.parameters(): p.requires_grad=False
    ps=[]
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad=True; mod.bias.requires_grad=True; ps += [mod.weight, mod.bias]
    return ps

def tta_mode(m, bn_mode):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout):
            mod.eval()
        elif isinstance(mod, nn.BatchNorm2d):
            if bn_mode=='bn_train':
                mod.train()
            else:
                # freeze running stats; still allow affine via requires_grad
                mod.eval()

def entropy(z):
    p=torch.sigmoid(z).clamp(1e-6,1-1e-6)
    return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)

def adapt(ld, method='tent', lr=1e-4, bn_mode='bn_freeze_stats', e_margin=0.4):
    m=load_fresh(); srcm=load_fresh(); srcm.eval(); opt=torch.optim.Adam(bn_affine(m), lr=lr)
    n_seen=n_kept=0; ys,pb,pa=[],[],[]
    for x,y in tqdm(ld, desc=f'{method} {bn_mode} lr={lr}'):
        x=x.to(device)
        with torch.no_grad(): pb.append(torch.sigmoid(srcm(x)).view(-1).cpu().numpy())
        tta_mode(m, bn_mode); n_seen += x.size(0)
        if method=='tent':
            opt.zero_grad(set_to_none=True); entropy(m(x)).mean().backward(); opt.step(); n_kept += x.size(0)
        else:
            with torch.no_grad(): keep=entropy(m(x))<e_margin
            k=int(keep.sum())
            if k>0:
                opt.zero_grad(set_to_none=True); entropy(m(x))[keep].mean().backward(); opt.step(); n_kept += k
        with torch.no_grad():
            m.eval(); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y,b,a=np.concatenate(ys),np.concatenate(pb),np.concatenate(pa)
    return dict(method=method, lr=lr, bn_mode=bn_mode, e_margin=None if method=='tent' else e_margin,
                coverage=n_kept/max(n_seen,1), before=metrics_of(y,b), after=metrics_of(y,a), harm=flip_table(y,b,a))

print('--- OOD val sweep (hospital 1). Source AUROC', round(ood_src['auroc'],4))
sweep=[]
for bn_mode in ('bn_freeze_stats', 'bn_train'):
    for lr in (1e-5, 3e-5, 5e-5, 1e-4):
        r=adapt(ood_loader,'tent',lr,bn_mode); r['split']='ood_val'; sweep.append(r)
        print(bn_mode, lr, 'AUROC', round(r['after']['auroc'],4),
              'F1', round(r['after']['f1'],4), 'harm', round(r['harm']['harm_rate'],4),
              'delta', round(r['after']['auroc']-ood_src['auroc'],4))

best=max(sweep, key=lambda r: r['after']['auroc'])
print('BEST Tent on val:', best['bn_mode'], best['lr'], 'AUROC', round(best['after']['auroc'],4))
collapsed = best['after']['auroc'] < ood_src['auroc'] - 0.02
print('collapsed_vs_source?', collapsed)

In [ ]:
print('--- EATA e_margin on OOD val at selected Tent lr/mode ---')
eata_sweep=[]
for em in (0.2, 0.4, 0.55):
    r=adapt(ood_loader,'eata',best['lr'],best['bn_mode'],e_margin=em); r['split']='ood_val'; eata_sweep.append(r)
    print('e_margin', em, 'AUROC', round(r['after']['auroc'],4), 'cov', round(r['coverage'],3), 'harm', round(r['harm']['harm_rate'],4))
best_eata=max(eata_sweep, key=lambda r: r['after']['auroc'])
print('BEST EATA on val e_margin', best_eata['e_margin'], 'AUROC', round(best_eata['after']['auroc'],4))

print('--- TEST once (frozen HPs) ---')
src_test=evaluate(load_fresh(), test_loader)
tent_test=adapt(test_loader,'tent',best['lr'],best['bn_mode']); tent_test['split']='test'
eata_test=adapt(test_loader,'eata',best['lr'],best['bn_mode'],e_margin=best_eata['e_margin']); eata_test['split']='test'
print('TEST source', round(src_test['auroc'],4), 'F1', round(src_test['f1'],4), 'ECE', round(src_test['ece'],4))
print('TEST tent  ', round(tent_test['after']['auroc'],4), 'F1', round(tent_test['after']['f1'],4), 'harm', round(tent_test['harm']['harm_rate'],4))
print('TEST eata  ', round(eata_test['after']['auroc'],4), 'cov', round(eata_test['coverage'],3), 'harm', round(eata_test['harm']['harm_rate'],4))

out={
    'cuda': True, 'device': torch.cuda.get_device_name(0),
    'checkpoint': str(CKPT_PATH), 'best_epoch': int(CKPT.get('epoch',3)),
    'source_ood': ood_src, 'source_test': src_test,
    'tent_selected': {'bn_mode': best['bn_mode'], 'lr': best['lr']},
    'eata_selected': {'bn_mode': best['bn_mode'], 'lr': best['lr'], 'e_margin': best_eata['e_margin']},
    'val_tent_sweep': sweep, 'val_eata_sweep': eata_sweep,
    'tent_test': tent_test, 'eata_test': eata_test,
    'note': 'HPs selected on OOD val only. BN freeze_stats = affine only, running stats frozen.',
}
p=WORK/'camelyon17_resnet18_tent_eata_s42.json'
p.write_text(json.dumps(out, indent=2))
print('wrote', p)